# VeReMi Paper 1 — Balanced Archive Split Optimizer

This notebook improves the already leakage-free archive split while **preserving whole-archive isolation**.

### Goal
Keep:
- Train = **157 archives**
- Validation = **34 archives**
- Test = **34 archives**
- zero archive overlap
- approximately the same number of archives from each attack stratum in each split

while reducing the large edge-level class-distribution differences observed in the initial split.

### Important
The optimizer **never splits an archive**. It only swaps complete archives between partitions.

### Input
Frozen graph dataset:
`GRAPH_DATASET_ROOT / "archives"`

Initial split:
`INITIAL_SPLIT_ROOT / "03_full_split_manifest.csv"`

### Output
`BALANCED_SPLIT_ROOT`


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import math

from src.config import GRAPH_DATASET_ROOT, INITIAL_SPLIT_ROOT, BALANCED_SPLIT_ROOT
DATA_ROOT = GRAPH_DATASET_ROOT / "archives"
INITIAL_SPLIT_DIR = INITIAL_SPLIT_ROOT
INITIAL_MANIFEST = INITIAL_SPLIT_DIR / "03_full_split_manifest.csv"

OUT = BALANCED_SPLIT_ROOT
OUT.mkdir(parents=True, exist_ok=True)

SEED = 20260811
RNG = np.random.default_rng(SEED)

SPLITS = ["train", "validation", "test"]
TARGET_ARCHIVE_COUNTS = {
    "train": 157,
    "validation": 34,
    "test": 34
}

manifest0 = pd.read_csv(INITIAL_MANIFEST)

print("Initial manifest rows:", len(manifest0))
display(pd.crosstab(manifest0["stratum"], manifest0["split"]))


## 1. Build exact per-archive edge-class counts

Each VeReMi archive contains benign traffic plus one attack type. We count every `attackerType` value per archive so the optimizer can balance **edge-level class proportions** without ever breaking archive boundaries.


In [ ]:
archive_dirs = {
    p.name: p
    for p in DATA_ROOT.iterdir()
    if p.is_dir()
}

assert len(archive_dirs) == 225

records = []

for i, row in manifest0.iterrows():
    if i == 0 or (i + 1) % 25 == 0:
        print(f"Counting archive labels {i+1}/225")

    name = row["archive_name"]
    p = archive_dirs[name] / "edge_features.parquet"

    edge = pd.read_parquet(
        p,
        columns=["attackerType"]
    )

    vc = edge["attackerType"].value_counts()

    rec = {
        "archive_name": name,
        "stratum": row["stratum"],
        "initial_split": row["split"],
        "edge_rows": len(edge)
    }

    for label in [0, 1, 2, 4, 8, 16]:
        rec[f"class_{label}"] = int(
            vc.get(label, 0)
        )

    records.append(rec)

archive_counts = pd.DataFrame(records)

archive_counts.to_csv(
    OUT / "01_archive_class_counts.csv",
    index=False
)

display(archive_counts.head())


In [ ]:
# Global class distribution
CLASS_LABELS = [0, 1, 2, 4, 8, 16]
CLASS_COLS = [f"class_{x}" for x in CLASS_LABELS]

global_counts = archive_counts[CLASS_COLS].sum().to_numpy(dtype=float)
global_total = global_counts.sum()
global_prop = global_counts / global_total

global_distribution = pd.DataFrame({
    "label": CLASS_LABELS,
    "edge_count": global_counts.astype("int64"),
    "fraction": global_prop
})

global_distribution.to_csv(
    OUT / "02_global_class_distribution.csv",
    index=False
)

display(global_distribution)


## 2. Preserve the initial per-stratum archive quotas

The current split already preserves attack-scenario representation at archive level. We keep those exact quotas and optimize **which archives** occupy each quota.

This means the optimizer cannot accidentally place too few complete attack scenarios in any partition.


In [ ]:
quota_table = pd.crosstab(
    manifest0["stratum"],
    manifest0["split"]
).reindex(columns=SPLITS, fill_value=0)

display(quota_table)

quota_table.to_csv(
    OUT / "03_fixed_stratum_quotas.csv"
)

print("\nGlobal split totals from quotas:")
print(quota_table.sum(axis=0))


## 3. Define the optimization objective

Two objectives are minimized simultaneously:

1. **Class-distribution error:** relative squared deviation of each split's six edge-level class proportions from the global distribution.
2. **Edge-mass error:** deviation of each split's total edge mass from the nominal 157/34/34 proportions.

Class balance receives the larger weight because the main concern is avoiding very small test/validation representation for individual attacks.


In [ ]:
# Numeric representation
df = archive_counts.copy()

split_to_idx = {
    "train": 0,
    "validation": 1,
    "test": 2
}
idx_to_split = {v:k for k,v in split_to_idx.items()}

X = df[CLASS_COLS].to_numpy(dtype=float)
edge_mass = X.sum(axis=1)

initial_assignment = (
    df["initial_split"]
    .map(split_to_idx)
    .to_numpy(dtype=int)
)

target_archive_frac = np.array([
    157/225,
    34/225,
    34/225
], dtype=float)

EPS = 1e-12

def aggregates(assignment):
    C = np.zeros((3, len(CLASS_LABELS)), dtype=float)
    for s in range(3):
        C[s] = X[assignment == s].sum(axis=0)
    return C

def score_from_counts(C):
    split_total = C.sum(axis=1)
    split_prop = C / split_total[:, None]

    # Relative class-proportion error.
    relative = (
        (split_prop - global_prop[None, :])
        /
        (global_prop[None, :] + EPS)
    )

    class_error = float(
        np.mean(relative ** 2)
    )

    edge_fraction = (
        split_total /
        split_total.sum()
    )

    edge_relative = (
        (edge_fraction - target_archive_frac)
        /
        target_archive_frac
    )

    edge_error = float(
        np.mean(edge_relative ** 2)
    )

    total_score = (
        0.90 * class_error
        +
        0.10 * edge_error
    )

    return total_score, class_error, edge_error


initial_C = aggregates(initial_assignment)
initial_score = score_from_counts(initial_C)

print("Initial total score:", initial_score[0])
print("Initial class error:", initial_score[1])
print("Initial edge-mass error:", initial_score[2])


## 4. Same-stratum swap optimizer

Only archives from the **same attack stratum** are swapped between partitions. This preserves every stratum quota exactly.

A simulated-annealing/local-search strategy is used with a fixed random seed for reproducibility.


In [ ]:
# Map each stratum to row indices.
stratum_indices = {
    s: g.index.to_numpy()
    for s, g in df.groupby("stratum")
}

def run_optimizer(
    starting_assignment,
    iterations=300_000,
    initial_temperature=0.0025,
    final_temperature=0.000001
):
    assignment = starting_assignment.copy()
    C = aggregates(assignment)

    current_score, _, _ = score_from_counts(C)

    best_assignment = assignment.copy()
    best_C = C.copy()
    best_score = current_score

    strata = list(stratum_indices.keys())

    for it in range(iterations):

        # Geometric temperature decay.
        progress = it / max(iterations - 1, 1)

        T = (
            initial_temperature
            *
            (
                final_temperature
                /
                initial_temperature
            ) ** progress
        )

        stratum = strata[
            RNG.integers(
                0,
                len(strata)
            )
        ]

        inds = stratum_indices[stratum]

        # Need two archives currently assigned to different splits.
        for _ in range(20):
            i, j = RNG.choice(
                inds,
                size=2,
                replace=False
            )
            si = assignment[i]
            sj = assignment[j]

            if si != sj:
                break
        else:
            continue

        # Candidate aggregate update.
        new_C = C.copy()

        new_C[si] -= X[i]
        new_C[si] += X[j]

        new_C[sj] -= X[j]
        new_C[sj] += X[i]

        new_score, _, _ = score_from_counts(
            new_C
        )

        delta = (
            new_score
            -
            current_score
        )

        accept = (
            delta <= 0
            or
            RNG.random()
            <
            math.exp(
                -delta / max(T, 1e-15)
            )
        )

        if accept:
            assignment[i], assignment[j] = (
                assignment[j],
                assignment[i]
            )

            C = new_C
            current_score = new_score

            if current_score < best_score:
                best_score = current_score
                best_assignment = assignment.copy()
                best_C = C.copy()

        if (
            (it + 1) % 50_000 == 0
        ):
            print(
                f"{it+1:,}/{iterations:,} "
                f"current={current_score:.8f} "
                f"best={best_score:.8f}"
            )

    return (
        best_assignment,
        best_C,
        best_score
    )


best_assignment, best_C, best_score = run_optimizer(
    initial_assignment
)

print("\nInitial score:", initial_score[0])
print("Optimized score:", best_score)
print(
    "Relative improvement:",
    100 * (
        initial_score[0]
        -
        best_score
    )
    /
    initial_score[0],
    "%"
)


## 5. Validate quotas and build the optimized manifest


In [ ]:
optimized = df[
    [
        "archive_name",
        "stratum",
        "edge_rows"
    ]
].copy()

optimized["split"] = [
    idx_to_split[x]
    for x in best_assignment
]

optimized["initial_split"] = df[
    "initial_split"
].values

optimized["changed_from_initial"] = (
    optimized["split"]
    !=
    optimized["initial_split"]
)

# Hard validation.
assert len(optimized) == 225
assert optimized["archive_name"].nunique() == 225

assert (
    optimized["split"]
    .value_counts()
    .to_dict()
    ==
    {
        "train": 157,
        "validation": 34,
        "test": 34
    }
)

optimized_quota = pd.crosstab(
    optimized["stratum"],
    optimized["split"]
).reindex(
    index=quota_table.index,
    columns=SPLITS,
    fill_value=0
)

assert (
    optimized_quota.values
    ==
    quota_table.values
).all()

print("Optimized split counts:")
display(
    optimized["split"]
    .value_counts()
    .to_frame("archives")
)

print(
    "Archives reassigned:",
    int(
        optimized[
            "changed_from_initial"
        ].sum()
    )
)

display(optimized_quota)


## 6. Compare initial and optimized class balance


In [ ]:
def distribution_table(assignment, version):
    C = aggregates(assignment)

    rows = []

    for s, split in enumerate(SPLITS):
        total = C[s].sum()

        for j, label in enumerate(CLASS_LABELS):
            rows.append({
                "version": version,
                "split": split,
                "label": label,
                "edge_count": int(C[s, j]),
                "fraction": (
                    C[s, j] / total
                    if total
                    else np.nan
                )
            })

    return pd.DataFrame(rows)


initial_dist = distribution_table(
    initial_assignment,
    "initial"
)

optimized_dist = distribution_table(
    best_assignment,
    "optimized"
)

comparison = pd.concat(
    [
        initial_dist,
        optimized_dist
    ],
    ignore_index=True
)

comparison.to_csv(
    OUT /
    "04_initial_vs_optimized_class_distribution.csv",
    index=False
)

display(
    comparison.pivot_table(
        index=["label"],
        columns=["version","split"],
        values="fraction"
    )
)


In [ ]:
# Quantify maximum class imbalance across splits.
def imbalance_summary(dist, version):
    pivot = dist.pivot(
        index="label",
        columns="split",
        values="fraction"
    )

    out = pd.DataFrame({
        "label": pivot.index,
        "minimum_fraction": pivot.min(axis=1).values,
        "maximum_fraction": pivot.max(axis=1).values,
        "range": (
            pivot.max(axis=1)
            -
            pivot.min(axis=1)
        ).values
    })

    out["version"] = version

    return out


imbalance = pd.concat(
    [
        imbalance_summary(
            initial_dist,
            "initial"
        ),
        imbalance_summary(
            optimized_dist,
            "optimized"
        )
    ],
    ignore_index=True
)

imbalance.to_csv(
    OUT /
    "05_class_imbalance_comparison.csv",
    index=False
)

display(imbalance)


## 7. Save optimized split manifests


In [ ]:
optimized = optimized.sort_values(
    ["split","stratum","archive_name"]
).reset_index(drop=True)

optimized.to_csv(
    OUT /
    "06_optimized_full_split_manifest.csv",
    index=False
)

for split, filename in [
    ("train", "train_archives_optimized.csv"),
    ("validation", "validation_archives_optimized.csv"),
    ("test", "test_archives_optimized.csv")
]:
    optimized[
        optimized["split"] == split
    ].to_csv(
        OUT / filename,
        index=False
    )


## 8. Compare objective scores and freeze only if improved


In [ ]:
optimized_score_tuple = score_from_counts(
    best_C
)

score_report = pd.DataFrame({
    "metric": [
        "total_objective",
        "class_distribution_error",
        "edge_mass_error"
    ],
    "initial": [
        initial_score[0],
        initial_score[1],
        initial_score[2]
    ],
    "optimized": [
        optimized_score_tuple[0],
        optimized_score_tuple[1],
        optimized_score_tuple[2]
    ]
})

score_report["improvement_percent"] = (
    100
    *
    (
        score_report["initial"]
        -
        score_report["optimized"]
    )
    /
    score_report["initial"]
)

score_report.to_csv(
    OUT /
    "07_split_objective_comparison.csv",
    index=False
)

display(score_report)


In [ ]:
# Final overlap / quota / improvement audit.

train = set(
    optimized.loc[
        optimized["split"] == "train",
        "archive_name"
    ]
)

val = set(
    optimized.loc[
        optimized["split"] == "validation",
        "archive_name"
    ]
)

test = set(
    optimized.loc[
        optimized["split"] == "test",
        "archive_name"
    ]
)

no_overlap = (
    len(train & val) == 0
    and
    len(train & test) == 0
    and
    len(val & test) == 0
)

quota_preserved = (
    optimized_quota.values
    ==
    quota_table.values
).all()

improved = (
    best_score
    <
    initial_score[0]
)

status = (
    "PASS_ADOPT_OPTIMIZED_SPLIT"
    if (
        no_overlap
        and
        quota_preserved
        and
        improved
    )
    else
    "REVIEW_REQUIRED"
)

final_report = pd.DataFrame({
    "check": [
        "Total archives",
        "Train archives",
        "Validation archives",
        "Test archives",
        "Archive overlap",
        "Per-stratum quotas preserved",
        "Objective improved",
        "Initial objective",
        "Optimized objective",
        "FINAL_STATUS"
    ],
    "value": [
        len(optimized),
        len(train),
        len(val),
        len(test),
        0 if no_overlap else 1,
        bool(quota_preserved),
        bool(improved),
        initial_score[0],
        best_score,
        status
    ]
})

final_report.to_csv(
    OUT /
    "08_FINAL_BALANCED_SPLIT_REPORT.csv",
    index=False
)

display(final_report)

print("\nFinal status:", status)
print("Outputs:", OUT)


## What to send back

Please upload or paste:

- `04_initial_vs_optimized_class_distribution.csv`
- `05_class_imbalance_comparison.csv`
- `07_split_objective_comparison.csv`
- `08_FINAL_BALANCED_SPLIT_REPORT.csv`
- `06_optimized_full_split_manifest.csv`

We will adopt the optimized split **only if it materially improves class balance while retaining 157/34/34 archives, zero overlap, and the original per-attack archive quotas**.
